# Pertemuan 12 - Asosiasi Data & Sistem Rekomendasi Dasar
Nama : Abdul Zaki Al-Muttaqin
Nim : 240401010042
Kelas : IF403

## Tujuan Praktikum
Praktikum ini bertujuan untuk memahami penerapan Association Rule Mining menggunakan algoritma Apriori serta memahami sistem rekomendasi sederhana dengan pendekatan Content-Based Filtering.

Pada praktikum ini dilakukan proses pencarian pola pembelian dari data transaksi, pembentukan aturan asosiasi menggunakan Support, Confidence, dan Lift, serta membandingkan hasil rekomendasi dari Association Rules dan Content-Based Filtering.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Generate & Eksplorasi Dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

produk = [
    'Roti', 'Selai', 'Susu', 'Sereal', 'Telur',
    'Keju', 'Kopi', 'Gula', 'Teh', 'Mentega'
]

# Buat 50 transaksi, setiap transaksi berisi 2-5 produk
transaksi = []

for _ in range(50):
    n_item = np.random.randint(2, 6)
    transaksi.append(
        list(np.random.choice(produk, n_item, replace=False))
    )

# Suntikkan pola: Roti sering bersama Selai
for i in range(20):
    if 'Roti' in transaksi[i] and 'Selai' not in transaksi[i]:
        transaksi[i].append('Selai')

print("Contoh transaksi:", transaksi[:3])
print("Jumlah transaksi:", len(transaksi))

Contoh transaksi: [[np.str_('Keju'), np.str_('Roti'), np.str_('Mentega'), np.str_('Kopi'), 'Selai'], [np.str_('Roti'), np.str_('Kopi'), np.str_('Teh'), np.str_('Selai'), np.str_('Mentega')], [np.str_('Kopi'), np.str_('Susu'), np.str_('Teh')]]
Jumlah transaksi: 50


Dataset transaksi berhasil dibuat sebanyak 50 transaksi.
Setiap transaksi terdiri dari 2 sampai 5 produk.
Pada dataset juga diberikan pola bahwa Roti sering dibeli bersama Selai,
sehingga pola tersebut diharapkan dapat ditemukan oleh algoritma Apriori.

In [ ]:
# One-Hot Encoding Transaksi
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()

te_array = te.fit(transaksi).transform(transaksi)

df = pd.DataFrame(te_array, columns=te.columns_)

print(df.head())

    Gula   Keju   Kopi  Mentega   Roti  Selai  Sereal   Susu    Teh  Telur
0  False   True   True     True   True   True   False  False  False  False
1  False  False   True     True   True   True   False  False   True  False
2  False  False   True    False  False  False   False   True   True  False
3  False   True  False    False  False   True   False  False   True   True
4   True   True  False     True  False  False   False   True  False  False


In [ ]:
# Cari Frequent Itemset dengan Apriori
from mlxtend.frequent_patterns import apriori

for ms in [0.05, 0.1, 0.2]:
    freq = apriori(
        df,
        min_support=ms,
        use_colnames=True
    )

    print(
        f"min_support={ms}: {len(freq)} itemset ditemukan"
    )

# Gunakan min_support 0.1
freq_items = apriori(
    df,
    min_support=0.1,
    use_colnames=True
)

freq_items = freq_items.sort_values(
    'support',
    ascending=False
)

print(freq_items.head(10))

min_support=0.05: 74 itemset ditemukan
min_support=0.1: 44 itemset ditemukan
min_support=0.2: 13 itemset ditemukan
    support      itemsets
5      0.52       (Selai)
8      0.46         (Teh)
3      0.42     (Mentega)
9      0.36       (Telur)
1      0.34        (Keju)
0      0.32        (Gula)
2      0.32        (Kopi)
4      0.32        (Roti)
7      0.32        (Susu)
36     0.24  (Selai, Teh)


In [ ]:
# Membentuk Aturan Asosiasi
from mlxtend.frequent_patterns import association_rules

rules = association_rules(
    freq_items,
    metric='confidence',
    min_threshold=0.5
)

rules = rules[
    rules['lift'] > 1
].sort_values(
    'lift',
    ascending=False
)

print(
    rules[
        ['antecedents', 'consequents',
         'support', 'confidence', 'lift']
    ].head(10)
)

         antecedents consequents  support  confidence      lift
10       (Teh, Keju)     (Telur)     0.12    0.857143  2.380952
15  (Selai, Mentega)      (Kopi)     0.10    0.625000  1.953125
11      (Gula, Roti)     (Selai)     0.10    1.000000  1.923077
7           (Sereal)   (Mentega)     0.14    0.777778  1.851852
8       (Telur, Teh)      (Keju)     0.12    0.600000  1.764706
13     (Kopi, Selai)   (Mentega)     0.10    0.714286  1.700680
9      (Telur, Keju)       (Teh)     0.12    0.750000  1.630435
12     (Gula, Selai)      (Roti)     0.10    0.500000  1.562500
14   (Kopi, Mentega)     (Selai)     0.10    0.714286  1.373626
1             (Roti)     (Selai)     0.22    0.687500  1.322115




Aturan asosiasi digunakan untuk mengetahui hubungan antarproduk.
Aturan dengan nilai lift lebih dari 1 menunjukkan bahwa kedua produk
memiliki hubungan positif dan lebih sering muncul bersama dibandingkan
jika terjadi secara kebetulan.

Aturan dengan nilai lift paling tinggi dapat dianggap sebagai aturan
yang memiliki hubungan paling kuat dalam dataset.

In [ ]:
# Content-Based Filtering
from sklearn.metrics.pairwise import cosine_similarity

katalog = pd.DataFrame({
    'produk': [
        'Roti', 'Susu', 'Sereal', 'Telur', 'Keju',
        'Kopi', 'Gula', 'Teh', 'Mentega', 'Selai'
    ],
    'kategori': [
        'Bakery', 'Bakery', 'Dairy', 'Bakery', 'Dairy',
        'Dairy', 'Minuman', 'Bumbu', 'Minuman', 'Dairy'
    ]
})

print(katalog)

    produk kategori
0     Roti   Bakery
1     Susu   Bakery
2   Sereal    Dairy
3    Telur   Bakery
4     Keju    Dairy
5     Kopi    Dairy
6     Gula  Minuman
7      Teh    Bumbu
8  Mentega  Minuman
9    Selai    Dairy


In [ ]:
fitur = pd.get_dummies(katalog['kategori'])

sim_matrix = cosine_similarity(fitur)

print("Matriks kemiripan:")
print(sim_matrix)

Matriks kemiripan:
[[1. 1. 0. 1. 0. 0. 0. 0. 0. 0.]
 [1. 1. 0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 1. 1. 0. 0. 0. 1.]
 [1. 1. 0. 1. 0. 0. 0. 0. 0. 0.]
 [0. 0. 1. 0. 1. 1. 0. 0. 0. 1.]
 [0. 0. 1. 0. 1. 1. 0. 0. 0. 1.]
 [0. 0. 0. 0. 0. 0. 1. 0. 1. 0.]
 [0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 0. 0. 1. 0. 1. 0.]
 [0. 0. 1. 0. 1. 1. 0. 0. 0. 1.]]


In [ ]:
# Fungsi rekomendasi
def rekomendasi_serupa(nama_produk, top_n=3):
    idx = katalog.index[
        katalog['produk'] == nama_produk
    ][0]

    skor = list(
        enumerate(sim_matrix[idx])
    )

    skor = sorted(
        skor,
        key=lambda x: x[1],
        reverse=True
    )

    skor = [
        s for s in skor
        if s[0] != idx
    ][:top_n]

    return katalog.iloc[
        [i for i, _ in skor]
    ][['produk', 'kategori']]

In [ ]:
print(
    "Mirip dengan Roti:"
)

print(
    rekomendasi_serupa('Roti')
)

Mirip dengan Roti:
   produk kategori
1    Susu   Bakery
3   Telur   Bakery
2  Sereal    Dairy




Content-Based Filtering memberikan rekomendasi berdasarkan kemiripan
atribut produk. Pada praktikum ini atribut yang digunakan adalah kategori.
Produk yang memiliki kategori yang sama akan memiliki tingkat kemiripan
yang lebih tinggi sehingga dapat direkomendasikan sebagai produk serupa.

In [ ]:
# Bandingkan Kedua Pendekatan
produk_target = 'Roti'

# Dari association rules:
# cari consequents dari aturan yang antecedent-nya mengandung Roti
rules_terkait = rules[
    rules['antecedents'].apply(
        lambda x: produk_target in x
    )
]

print("Rekomendasi dari Association Rules:")
print(
    rules_terkait[
        ['consequents', 'lift']
    ].head()
)

print("\nRekomendasi dari Content-Based:")
print(
    rekomendasi_serupa(produk_target)
)

Rekomendasi dari Association Rules:
   consequents      lift
11     (Selai)  1.923077
1      (Selai)  1.322115

Rekomendasi dari Content-Based:
   produk kategori
1    Susu   Bakery
3   Telur   Bakery
2  Sereal    Dairy


# Kesimpulan

Pada praktikum ini saya mempelajari cara menemukan pola pembelian menggunakan algoritma Apriori dan membuat aturan asosiasi berdasarkan nilai Support, Confidence, dan Lift. Selain itu, saya juga memahami cara membuat rekomendasi produk menggunakan Content-Based Filtering.

Dari hasil praktikum, Association Rule Mining berhasil menemukan hubungan antara produk, salah satunya Roti yang memiliki hubungan dengan Selai. Sedangkan Content-Based Filtering memberikan rekomendasi berdasarkan kemiripan kategori produk.

Keterbatasannya, data transaksi yang digunakan masih berupa data sederhana dan jumlahnya terbatas, sehingga hasil aturan asosiasi dan rekomendasi belum tentu sama jika diterapkan pada data transaksi yang lebih besar dan nyata.